# Custom Triton Kernels — GPU Programming in Python

**Module 25** of the PyTorch Complete Learning Guide

| | |
|---|---|
| **Prerequisites** | Module 07 (Training), Module 08 (torch.compile) |
| **Time** | ~3 hours |
| **GPU Required** | Yes (NVIDIA, Compute Capability 7.0+) |

In this notebook we learn to write custom GPU kernels in Python using **Triton**, OpenAI's GPU programming language. We cover the programming model, write practical kernels (vector add, fused add+ReLU, fused softmax), benchmark them against PyTorch, and integrate them with `torch.library` and `torch.compile`.

> **Note:** Triton kernels require an NVIDIA GPU. Cells are marked with 🔷 (GPU required) or ⬜ (CPU-safe). On CPU-only machines the GPU cells will print explanations but not run kernels.

## 1. What is Triton?

**Triton** is OpenAI's open-source language and compiler for writing GPU kernels in Python.

```
Ease of use:   PyTorch  >  Triton  >  CUDA C++
Performance:   CUDA C++ ≈  Triton  >  PyTorch (eager)
```

Key facts:
- **Python syntax** with NumPy-like operations, compiled to PTX/SASS for NVIDIA GPUs
- **Near-peak performance** — the compiler handles tiling, shared memory, register allocation
- **PyTorch uses Triton internally** — TorchInductor generates Triton code from `torch.compile`
- **Custom kernel integration** — register your kernels with `torch.library` for autograd + compile support

## 2. Setup & Imports

In [ ]:
import torch
import time

HAS_TRITON = False
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    print("Triton not installed. Install with: pip install triton")

HAS_CUDA = torch.cuda.is_available()

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {HAS_CUDA}")
print(f"Triton available: {HAS_TRITON}")
if HAS_CUDA:
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

## 3. Triton Programming Model

Triton programs run as a **grid of blocks** ("programs"). Each block processes a chunk of data:

```
Data:    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

Block 0: [0, 1, 2, 3]          pid=0, processes indices 0-3
Block 1: [4, 5, 6, 7]          pid=1, processes indices 4-7
Block 2: [8, 9, 10, 11]        pid=2, processes indices 8-11
Block 3: [12, 13, 14, 15]      pid=3, processes indices 12-15
```

| Concept | Description |
|---------|-------------|
| `@triton.jit` | Compiles Python function → GPU kernel |
| `tl.program_id(axis)` | Current block index |
| `BLOCK_SIZE: tl.constexpr` | Elements per block (compile-time constant) |
| `tl.arange(0, N)` | Range [0..N-1] within a block |
| `tl.load(ptr, mask)` | Load from GPU memory |
| `tl.store(ptr, val, mask)` | Store to GPU memory |
| Grid | `kernel[grid](args...)` — total number of blocks |

## 4. Vector Addition Kernel (Line-by-Line) 🔷

The simplest Triton kernel — adding two vectors element by element.

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @triton.jit
    def add_kernel(
        x_ptr,      # pointer to first input
        y_ptr,      # pointer to second input
        out_ptr,    # pointer to output
        n,          # total number of elements
        BLOCK_SIZE: tl.constexpr,
    ):
        # Which block am I? (0-indexed)
        pid = tl.program_id(0)

        # Global indices this block handles
        # Block 0 → [0..BS-1], Block 1 → [BS..2*BS-1], ...
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

        # Mask: True for valid indices (last block may be partial)
        mask = offsets < n

        # Load from GPU memory (masked-out lanes default to 0)
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)

        # Compute in registers — no memory traffic
        result = x + y

        # Write back (masked stores skip out-of-bounds)
        tl.store(out_ptr + offsets, result, mask=mask)

    def triton_add(x, y):
        output = torch.empty_like(x)
        n = x.numel()
        BLOCK_SIZE = 1024
        grid = (triton.cdiv(n, BLOCK_SIZE),)
        add_kernel[grid](x, y, output, n, BLOCK_SIZE=BLOCK_SIZE)
        return output

    # Test
    x = torch.randn(100_000, device='cuda')
    y = torch.randn(100_000, device='cuda')
    z = triton_add(x, y)
    print(f"Max error: {(z - (x + y)).abs().max().item():.2e}")
    print("Vector addition: Triton matches PyTorch ✓")
else:
    print("[GPU required] Vector add kernel pattern:")
    print("  pid = tl.program_id(0)")
    print("  offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)")
    print("  x = tl.load(x_ptr + offsets, mask=offsets < n)")
    print("  tl.store(out_ptr + offsets, x + y, mask=offsets < n)")

## 5. Fused Add + ReLU — Why Fusion Matters 🔷

### The Memory Bandwidth Problem

```
Unfused (2 kernels):                  Fused (1 kernel):
┌─────────┐  ┌─────────┐             ┌─────────┐
│ Read x  │  │Read temp│             │ Read x  │
│ Read y  │  │         │             │ Read y  │
│ Write   │  │ Write   │             │ Write   │
│  temp   │  │  out    │             │  out    │
└─────────┘  └─────────┘             └─────────┘
 3 mem ops +  2 mem ops    =           3 mem ops total
 = 5 mem ops total                     40% less traffic!
```

Fusion eliminates the temporary tensor and its associated memory read/write.

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @triton.jit
    def fused_add_relu_kernel(
        x_ptr, y_ptr, out_ptr, n,
        BLOCK_SIZE: tl.constexpr,
    ):
        pid = tl.program_id(0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        # Fused: add + relu in one pass (no intermediate write!)
        result = tl.maximum(x + y, 0.0)
        tl.store(out_ptr + offsets, result, mask=mask)

    def triton_add_relu(x, y):
        output = torch.empty_like(x)
        n = x.numel()
        BLOCK_SIZE = 1024
        grid = (triton.cdiv(n, BLOCK_SIZE),)
        fused_add_relu_kernel[grid](x, y, output, n, BLOCK_SIZE=BLOCK_SIZE)
        return output

    # Verify correctness
    x = torch.randn(50_000, device='cuda')
    y = torch.randn(50_000, device='cuda')
    out_triton = triton_add_relu(x, y)
    out_pytorch = torch.relu(x + y)
    print(f"Max error: {(out_triton - out_pytorch).abs().max().item():.2e}")
    print("Fused add+relu: Triton matches PyTorch ✓")
else:
    print("[GPU required] Key line: result = tl.maximum(x + y, 0.0)")
    print("x+y computed in registers, immediately passed to max — no temp tensor.")

## 6. Fused Softmax Kernel 🔷

A practical kernel: compute softmax over each row in a single pass.

**Algorithm** (per row):
1. `max_val = max(row)` — numerical stability
2. `row = row - max_val` — shift
3. `row = exp(row)` — exponentiate
4. `sum_val = sum(row)` — normalization constant
5. `out = row / sum_val` — normalize

All 5 steps happen in registers — the row is loaded once and written once.

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @triton.jit
    def softmax_kernel(
        input_ptr, output_ptr,
        n_cols,
        input_row_stride, output_row_stride,
        BLOCK_SIZE: tl.constexpr,
    ):
        row_idx = tl.program_id(0)  # one block per row
        row_start = input_ptr + row_idx * input_row_stride
        col_offsets = tl.arange(0, BLOCK_SIZE)
        mask = col_offsets < n_cols

        # Load row (-inf for masked positions so exp(-inf)=0)
        row = tl.load(row_start + col_offsets, mask=mask, other=float('-inf'))

        # Numerically stable softmax
        row_max = tl.max(row, axis=0)
        numerator = tl.exp(row - row_max)
        denominator = tl.sum(numerator, axis=0)
        softmax_out = numerator / denominator

        out_start = output_ptr + row_idx * output_row_stride
        tl.store(out_start + col_offsets, softmax_out, mask=mask)

    def triton_softmax(x):
        n_rows, n_cols = x.shape
        BLOCK_SIZE = triton.next_power_of_2(n_cols)
        output = torch.empty_like(x)
        softmax_kernel[(n_rows,)](
            x, output, n_cols,
            x.stride(0), output.stride(0),
            BLOCK_SIZE=BLOCK_SIZE,
        )
        return output

    # Test
    x = torch.randn(512, 256, device='cuda')
    out_t = triton_softmax(x)
    out_p = torch.softmax(x, dim=-1)
    print(f"Max error: {(out_t - out_p).abs().max().item():.2e}")
    print(f"Row sums (should be ~1.0): {out_t.sum(dim=-1)[:5].tolist()}")
    print("Fused softmax: Triton matches PyTorch ✓")
else:
    print("[GPU required] Softmax kernel: one block per row")
    print("  Load row → max → exp(row-max) → sum → divide → store")
    print("  All in registers, one read + one write.")

## 7. Benchmarking: Triton vs PyTorch 🔷

Let's measure the actual speedup from our fused kernels.

In [ ]:
def benchmark_fn(fn, *args, warmup=25, rep=100):
    """Simple GPU benchmark returning median time in ms."""
    for _ in range(warmup):
        fn(*args)
    if HAS_CUDA:
        torch.cuda.synchronize()
    times = []
    for _ in range(rep):
        start = time.perf_counter()
        fn(*args)
        if HAS_CUDA:
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    times.sort()
    return times[len(times) // 2]

if HAS_TRITON and HAS_CUDA:
    print("=== Vector Addition ===")
    for sz in [100_000, 1_000_000, 10_000_000]:
        x = torch.randn(sz, device='cuda')
        y = torch.randn(sz, device='cuda')
        t_pt = benchmark_fn(torch.add, x, y)
        t_tr = benchmark_fn(triton_add, x, y)
        print(f"  N={sz:>10,d}: PyTorch={t_pt:.3f}ms  Triton={t_tr:.3f}ms  Speedup={t_pt/t_tr:.2f}x")

    print("\n=== Fused Add + ReLU ===")
    for sz in [100_000, 1_000_000, 10_000_000]:
        x = torch.randn(sz, device='cuda')
        y = torch.randn(sz, device='cuda')
        t_pt = benchmark_fn(lambda a, b: torch.relu(a + b), x, y)
        t_tr = benchmark_fn(triton_add_relu, x, y)
        print(f"  N={sz:>10,d}: PyTorch={t_pt:.3f}ms  Triton={t_tr:.3f}ms  Speedup={t_pt/t_tr:.2f}x")

    print("\n=== Fused Softmax ===")
    for m, k in [(512, 256), (2048, 512), (4096, 1024)]:
        x = torch.randn(m, k, device='cuda')
        t_pt = benchmark_fn(lambda t: torch.softmax(t, dim=-1), x)
        t_tr = benchmark_fn(triton_softmax, x)
        print(f"  ({m}x{k}): PyTorch={t_pt:.3f}ms  Triton={t_tr:.3f}ms  Speedup={t_pt/t_tr:.2f}x")
else:
    print("[GPU required for benchmarks]")
    print("Typical speedups on A100:")
    print("  Vector add: ~1.0-1.2x (compute-bound, not much to fuse)")
    print("  Fused add+relu: ~1.3-1.5x (saves one memory round-trip)")
    print("  Fused softmax: ~1.5-3.0x (saves multiple intermediates)")

## 8. Grid Launch Patterns

The **grid** tells Triton how many blocks to launch.

| Pattern | Code | Use Case |
|---------|------|----------|
| 1D static | `grid = (cdiv(n, BS),)` | Elementwise ops |
| 2D static | `grid = (cdiv(M, BM), cdiv(N, BN))` | Matmul, 2D tiling |
| Lambda | `grid = lambda meta: (cdiv(n, meta['BS']),)` | With `@triton.autotune` |

In [ ]:
if HAS_TRITON:
    n = 100_000
    BS = 1024
    grid_1d = (triton.cdiv(n, BS),)
    print(f"1D grid: n={n:,}, BS={BS} → grid={grid_1d} → {grid_1d[0]} blocks")

    M, N, BM, BN = 4096, 4096, 128, 128
    grid_2d = (triton.cdiv(M, BM), triton.cdiv(N, BN))
    print(f"2D grid: ({M}x{N}), BM={BM}, BN={BN} → grid={grid_2d} → {grid_2d[0]*grid_2d[1]} blocks")

    # Lambda grid (used with autotuning)
    grid_fn = lambda meta: (triton.cdiv(n, meta['BLOCK_SIZE']),)
    print(f"Lambda grid: meta={{'BLOCK_SIZE': 512}} → grid={grid_fn({'BLOCK_SIZE': 512})}")
else:
    print("1D: grid = (ceil(n / BLOCK_SIZE),)")
    print("2D: grid = (ceil(M / BLOCK_M), ceil(N / BLOCK_N))")
    print("Lambda: grid = lambda meta: (ceil(n / meta['BLOCK_SIZE']),)")

## 9. Integrating with torch.library 🔷

To make a Triton kernel work with **autograd**, **torch.compile**, and **torch.export**, register it as a PyTorch custom op:

```
Triton kernel (@triton.jit)
    ↓
@torch.library.custom_op    — eager implementation
    ↓
.register_fake              — shape inference for compile
    ↓
.register_autograd          — backward pass for training
    ↓
Works with torch.compile ✓
```

In [ ]:
if HAS_TRITON and HAS_CUDA:
    # Step 1: Triton kernel
    @triton.jit
    def _gelu_kernel(x_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
        pid = tl.program_id(0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n
        x = tl.load(x_ptr + offsets, mask=mask)
        k = 0.7978845608028654  # sqrt(2/pi)
        out = 0.5 * x * (1.0 + tl.math.tanh(k * (x + 0.044715 * x * x * x)))
        tl.store(out_ptr + offsets, out, mask=mask)

    # Step 2: Register as custom op
    @torch.library.custom_op("nb::triton_gelu", mutates_args=())
    def triton_gelu(x: torch.Tensor) -> torch.Tensor:
        output = torch.empty_like(x)
        n = x.numel()
        grid = (triton.cdiv(n, 1024),)
        _gelu_kernel[grid](x, output, n, BLOCK_SIZE=1024)
        return output

    # Step 3: Meta implementation (shape inference)
    @triton_gelu.register_fake
    def triton_gelu_fake(x):
        return torch.empty_like(x)

    # Step 4: Autograd
    def gelu_setup(ctx, inputs, output):
        ctx.save_for_backward(inputs[0])

    def gelu_backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        k = 0.7978845608028654
        inner = k * (x + 0.044715 * x**3)
        tanh_inner = torch.tanh(inner)
        grad = 0.5 * (1 + tanh_inner) + 0.5 * x * (1 - tanh_inner**2) * k * (1 + 3 * 0.044715 * x**2)
        return grad_out * grad

    triton_gelu.register_autograd(gelu_backward, setup_context=gelu_setup)

    # Test forward
    x = torch.randn(2048, device='cuda')
    y = torch.ops.nb.triton_gelu(x)
    y_ref = torch.nn.functional.gelu(x, approximate='tanh')
    print(f"Forward max error: {(y - y_ref).abs().max().item():.2e}")

    # Test backward
    x_g = torch.randn(512, device='cuda', requires_grad=True)
    torch.ops.nb.triton_gelu(x_g).sum().backward()
    print(f"Backward: gradient computed ✓ (shape {x_g.grad.shape})")
    print("Custom op with autograd: works ✓")
else:
    print("[GPU required] Registration pattern:")
    print("  @torch.library.custom_op('ns::name', mutates_args=())")
    print("  def my_op(x: Tensor) -> Tensor: ...")
    print("  @my_op.register_fake")
    print("  def my_op_fake(x): return torch.empty_like(x)")

## 10. Custom Op + torch.compile 🔷

Once registered with `register_fake`, your custom op works seamlessly with `torch.compile`.

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @torch.compile
    def compiled_fn(x):
        return torch.ops.nb.triton_gelu(x)

    x = torch.randn(4096, device='cuda')
    y = compiled_fn(x)
    y_ref = torch.nn.functional.gelu(x, approximate='tanh')
    print(f"torch.compile + custom op: max error = {(y - y_ref).abs().max().item():.2e}")
    print("Works with torch.compile ✓")
else:
    print("[GPU required] @torch.compile + custom op = seamless integration")

## 11. Autotuning with @triton.autotune 🔷

Different problem sizes perform best with different block sizes. `@triton.autotune` benchmarks multiple configs and picks the fastest:

```python
@triton.autotune(
    configs=[triton.Config({'BLOCK_SIZE': 128}),
             triton.Config({'BLOCK_SIZE': 256}),
             triton.Config({'BLOCK_SIZE': 1024})],
    key=['n'],  # re-tune when n changes
)
@triton.jit
def my_kernel(..., BLOCK_SIZE: tl.constexpr): ...
```

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @triton.autotune(
        configs=[
            triton.Config({'BLOCK_SIZE': 128}),
            triton.Config({'BLOCK_SIZE': 256}),
            triton.Config({'BLOCK_SIZE': 512}),
            triton.Config({'BLOCK_SIZE': 1024}),
            triton.Config({'BLOCK_SIZE': 2048}),
        ],
        key=['n'],
    )
    @triton.jit
    def add_autotuned(x_ptr, y_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
        pid = tl.program_id(0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        tl.store(out_ptr + offsets, x + y, mask=mask)

    x = torch.randn(1_000_000, device='cuda')
    y = torch.randn(1_000_000, device='cuda')
    output = torch.empty_like(x)
    grid = lambda meta: (triton.cdiv(1_000_000, meta['BLOCK_SIZE']),)
    print("Running autotuned kernel (first call benchmarks all configs)...")
    add_autotuned[grid](x, y, output, 1_000_000)
    print(f"Result correct: {torch.allclose(output, x + y, atol=1e-6)}")
    if hasattr(add_autotuned, 'best_config'):
        print(f"Best config selected: {add_autotuned.best_config}")
else:
    print("[GPU required] Autotuning benchmarks configs and caches the best one.")

## 12. Matrix Multiplication (Tiled) 🔷

Matrix multiply demonstrates 2D tiling. Each block computes a `BLOCK_M x BLOCK_N` tile of the output, iterating over K in chunks of `BLOCK_K`:

```
C[i, j] = sum_k A[i, k] * B[k, j]

Each block:
  acc = zeros(BLOCK_M, BLOCK_N)
  for k in range(0, K, BLOCK_K):
      acc += A_tile @ B_tile     # tl.dot uses Tensor Cores
  C_tile = acc
```

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @triton.jit
    def matmul_kernel(
        a_ptr, b_ptr, c_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    ):
        pid_m = tl.program_id(0)
        pid_n = tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
        for k in range(0, K, BLOCK_K):
            offs_k = k + tl.arange(0, BLOCK_K)
            a = tl.load(a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak,
                        mask=(offs_m[:, None] < M) & (offs_k[None, :] < K), other=0.0)
            b = tl.load(b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn,
                        mask=(offs_k[:, None] < K) & (offs_n[None, :] < N), other=0.0)
            acc += tl.dot(a, b)
        tl.store(c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn,
                 acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

    def triton_matmul(a, b):
        M, K = a.shape
        K2, N = b.shape
        assert K == K2
        c = torch.empty((M, N), device=a.device, dtype=torch.float32)
        BM, BN, BK = 64, 64, 32
        grid = (triton.cdiv(M, BM), triton.cdiv(N, BN))
        matmul_kernel[grid](a, b, c, M, N, K,
                            a.stride(0), a.stride(1),
                            b.stride(0), b.stride(1),
                            c.stride(0), c.stride(1),
                            BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK)
        return c

    M, K, N = 256, 512, 256
    a = torch.randn(M, K, device='cuda', dtype=torch.float32)
    b = torch.randn(K, N, device='cuda', dtype=torch.float32)
    c_triton = triton_matmul(a, b)
    c_torch = a @ b
    print(f"Matmul ({M}x{K}) @ ({K}x{N}): max error = {(c_triton - c_torch).abs().max().item():.2e}")
    print("Tiled matmul: correct ✓")
else:
    print("[GPU required] Tiled matmul uses tl.dot for Tensor Core acceleration.")
    print("Each block computes a BLOCK_M x BLOCK_N tile, looping over K in BLOCK_K chunks.")

## 13. How TorchInductor Uses Triton

When you `torch.compile` a function, TorchInductor:

1. **Traces** with Dynamo → FX graph
2. **Fuses** compatible operations
3. **Generates Triton kernels** for each fused group
4. **Compiles** to PTX and caches

View generated code with:
```bash
TORCH_LOGS="output_code" python my_script.py
```

Or programmatically:
```python
import torch._logging
torch._logging.set_logs(output_code=True)
```

In [ ]:
if HAS_CUDA:
    @torch.compile
    def inductor_fused(x, y):
        return torch.relu(x + y)

    x = torch.randn(2048, device='cuda')
    y = torch.randn(2048, device='cuda')
    result = inductor_fused(x, y)
    print(f"Inductor auto-fused add+relu: correct = {torch.allclose(result, torch.relu(x + y))}")
    print("\nInductor generates Triton code that looks like:")
    print("  tmp0 = tl.load(in_ptr0 + xindex, xmask)")
    print("  tmp1 = tl.load(in_ptr1 + xindex, xmask)")
    print("  tmp2 = tmp0 + tmp1")
    print("  tmp3 = tl.maximum(tmp2, 0)")
    print("  tl.store(out_ptr0 + xindex, tmp3, xmask)")
    print("\n→ Same fusion we wrote by hand in section 5!")
else:
    print("Inductor automatically fuses operations into Triton kernels.")
    print("Use TORCH_LOGS='output_code' to see what it generates.")

## 14. Common Patterns

### Reduction (Single Block)
```python
x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
total = tl.sum(x, axis=0)
tl.store(out_ptr, total)
```

### Elementwise with Broadcast
```python
x = tl.load(x_ptr + offsets, mask=mask)
bias = tl.load(bias_ptr + offsets % hidden_dim, mask=mask)
out = tl.maximum(x + bias, 0.0)
```

### Random Numbers (for Dropout)
```python
random = tl.rand(seed, offsets)
keep = random > p_drop
x = tl.where(keep, x / (1 - p_drop), 0.0)
```

## 15. Triton vs CUDA

| Aspect | Triton | CUDA C++ |
|--------|--------|----------|
| Language | Python | C++ |
| Shared memory | Automatic | Manual |
| Thread control | Block-level | Full warp/thread |
| Performance | ~80-95% of CUDA | 100% |
| Tensor Cores | `tl.dot` (auto) | `wmma`/`mma.sync` (manual) |
| Iteration speed | Fast | Slow |

**Use Triton for:** elementwise, reductions, matmul, softmax, normalization, attention

**Use CUDA for:** warp shuffles, cooperative groups, custom memory patterns, extreme tuning

## 16. Complete Example: Fused RMSNorm 🔷

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @triton.jit
    def _rmsnorm_kernel(
        x_ptr, w_ptr, out_ptr,
        n_cols, eps,
        x_stride, out_stride,
        BLOCK: tl.constexpr,
    ):
        row = tl.program_id(0)
        cols = tl.arange(0, BLOCK)
        mask = cols < n_cols
        x = tl.load(x_ptr + row * x_stride + cols, mask=mask, other=0.0).to(tl.float32)
        rms = tl.math.rsqrt(tl.sum(x * x, axis=0) / n_cols + eps)
        w = tl.load(w_ptr + cols, mask=mask, other=1.0)
        tl.store(out_ptr + row * out_stride + cols, x * rms * w, mask=mask)

    def triton_rmsnorm(x, weight, eps=1e-6):
        n_rows, n_cols = x.shape
        out = torch.empty_like(x)
        BS = triton.next_power_of_2(n_cols)
        _rmsnorm_kernel[(n_rows,)](x, weight, out, n_cols, eps,
                                   x.stride(0), out.stride(0), BLOCK=BS)
        return out

    # Compare with PyTorch
    M, D = 256, 768
    x = torch.randn(M, D, device='cuda')
    w = torch.ones(D, device='cuda')
    out_t = triton_rmsnorm(x, w)
    rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + 1e-6)
    out_ref = (x.float() * rms * w).to(x.dtype)
    print(f"Fused RMSNorm ({M}x{D}): max error = {(out_t - out_ref).abs().max().item():.2e}")
    print("Fused RMSNorm: correct ✓")
else:
    print("[GPU required] RMSNorm: load row → mean(x^2) → rsqrt → scale → store")
    print("All in one kernel — no intermediate tensors.")

## 17. Upstream Updates (June 2026)

Recent PyTorch commits relevant to Triton kernel development:

| PR | Area | Summary |
|----|------|---------|
| #187402 | Optimizers | Muon optimizer: `spectral_unclamped` scaling |
| #186300 | Distributed | c10d abort hooks, pre/post collective hooks |
| #187387 | Distributed | Public `torch.distributed.set_timeout` API |
| #183838 | Inductor | Unbacked FlexAttention predicates |
| #187406 | Testing | torchfuzz ~190 ops coverage expansion |
| #186976 | Dynamo | `object()` sentinel support (fewer graph breaks) |

---

## Exercise: Fused LayerNorm Triton Kernel

**Goal:** Write a Triton kernel that computes LayerNorm in a single pass.

**Algorithm** (per row):
1. Compute mean: `mu = sum(x) / n`
2. Compute variance: `var = sum((x - mu)^2) / n`
3. Normalize: `x_hat = (x - mu) / sqrt(var + eps)`
4. Scale and shift: `out = gamma * x_hat + beta`

**Skeleton:**

In [ ]:
# Exercise: Fill in the TODOs

if HAS_TRITON and HAS_CUDA:
    @triton.jit
    def layernorm_kernel(
        x_ptr, gamma_ptr, beta_ptr, out_ptr,
        n_cols, eps,
        x_stride, out_stride,
        BLOCK: tl.constexpr,
    ):
        row = tl.program_id(0)
        cols = tl.arange(0, BLOCK)
        mask = cols < n_cols

        x = tl.load(x_ptr + row * x_stride + cols, mask=mask, other=0.0).to(tl.float32)

        # TODO: Compute mean
        # mean = ...

        # TODO: Compute variance
        # var = ...

        # TODO: Normalize
        # x_hat = ...

        # TODO: Scale and shift
        # gamma = tl.load(...)
        # beta = tl.load(...)
        # out = ...

        # tl.store(out_ptr + row * out_stride + cols, out, mask=mask)
        pass

    print("Fill in the TODOs above to implement fused LayerNorm!")
    print("Hints:")
    print("  - mean = tl.sum(x, axis=0) / n_cols")
    print("  - var = tl.sum((x - mean) * (x - mean), axis=0) / n_cols")
    print("  - x_hat = (x - mean) * tl.math.rsqrt(var + eps)")
else:
    print("[GPU required for exercise]")
    print("Solution approach:")
    print("  1. Load row into registers")
    print("  2. mean = tl.sum(x) / n_cols")
    print("  3. var = tl.sum((x - mean)^2) / n_cols")
    print("  4. x_hat = (x - mean) * rsqrt(var + eps)")
    print("  5. out = gamma * x_hat + beta")

### Exercise Solution

In [ ]:
if HAS_TRITON and HAS_CUDA:
    @triton.jit
    def layernorm_solution(
        x_ptr, gamma_ptr, beta_ptr, out_ptr,
        n_cols, eps,
        x_stride, out_stride,
        BLOCK: tl.constexpr,
    ):
        row = tl.program_id(0)
        cols = tl.arange(0, BLOCK)
        mask = cols < n_cols

        x = tl.load(x_ptr + row * x_stride + cols, mask=mask, other=0.0).to(tl.float32)

        mean = tl.sum(x, axis=0) / n_cols
        diff = x - mean
        var = tl.sum(diff * diff, axis=0) / n_cols
        x_hat = diff * tl.math.rsqrt(var + eps)

        gamma = tl.load(gamma_ptr + cols, mask=mask, other=1.0)
        beta = tl.load(beta_ptr + cols, mask=mask, other=0.0)
        out = gamma * x_hat + beta

        tl.store(out_ptr + row * out_stride + cols, out, mask=mask)

    def triton_layernorm(x, gamma, beta, eps=1e-5):
        n_rows, n_cols = x.shape
        out = torch.empty_like(x)
        BS = triton.next_power_of_2(n_cols)
        layernorm_solution[(n_rows,)](x, gamma, beta, out, n_cols, eps,
                                      x.stride(0), out.stride(0), BLOCK=BS)
        return out

    M, D = 128, 512
    x = torch.randn(M, D, device='cuda')
    gamma = torch.ones(D, device='cuda')
    beta = torch.zeros(D, device='cuda')
    out_t = triton_layernorm(x, gamma, beta)
    ln = torch.nn.LayerNorm(D, device='cuda')
    out_ref = ln(x)
    print(f"Fused LayerNorm ({M}x{D}): max error = {(out_t - out_ref).abs().max().item():.2e}")
    print("LayerNorm exercise: correct ✓")
else:
    print("[GPU required] See solution pattern above.")

---

## Key Takeaways

1. **Triton** lets you write GPU kernels in Python with ~80-95% of CUDA performance
2. **Block-based model**: grid of programs, each processing `BLOCK_SIZE` elements via `program_id`
3. **Fusion is the killer feature**: combine multiple ops into one kernel to cut memory traffic
4. **Practical kernels**: fused softmax, RMSNorm, and LayerNorm show real-world benefits
5. **PyTorch integration**: `torch.library.custom_op` → `register_fake` → `register_autograd`
6. **Autotuning**: `@triton.autotune` finds the best block size for your GPU and problem size
7. **TorchInductor generates Triton**: check with `TORCH_LOGS="output_code"` before writing custom kernels
8. **Check torch.compile first**: often Inductor already fuses your operations optimally

---

*Module 25 of the [PyTorch Complete Learning Guide](../README.md)*